# H0 vs OVR LR 오류 다양성 감사

새 blend나 제출을 만들지 않습니다. 최종 H0와 동일 structured+EB 입력의 OVR LR이 H0의 오답을 실질적으로 회복하는지만 확인합니다.

In [ ]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
RUNNER = ROOT / 'experiments/gs/notebooks/exp_model_010/common/run_h0_ovr_diversity_audit.py'
RESULT = ROOT / 'experiments/gs/notebooks/exp_model_010/result'
RUN_ID = 'exp-h0-ovr-diversity-audit-01'
RUN_EXPERIMENT = False
assert RUNNER.exists()

In [ ]:
subprocess.run([sys.executable, str(RUNNER), '--smoke'], check=True)

In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen([sys.executable, str(RUNNER), '--run-id', RUN_ID], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc='H0 / OVR diversity audit', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('OVR diversity runner failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: 결과가 있다면 집계만 수행합니다.')

In [ ]:
summary = pd.read_csv(RESULT / f'{RUN_ID}_seed42_summary.csv')
folds = pd.read_csv(RESULT / f'{RUN_ID}_seed42_fold_metrics.csv')
classes = pd.read_csv(RESULT / f'{RUN_ID}_seed42_class_metrics.csv')
errors = pd.read_csv(RESULT / f'{RUN_ID}_seed42_error_overlap.csv')
audit = json.loads((RESULT / f'{RUN_ID}_seed42_diversity_audit.json').read_text())
assert summary.leakage_check.all() and summary.nan_as_mutation_count.eq(0).all()
display(summary)
display(folds.pivot(index='fold', columns='variant', values='macro_f1'))
display(pd.DataFrame([audit])[['hard_prediction_disagreement_rate', 'h0_wrong_ovr_recovered_count', 'h0_correct_ovr_broken_count', 'oracle_macro_f1_diagnostic_only', 'ovr_diversity_candidate']])

In [ ]:
folds.pivot(index='fold', columns='variant', values='macro_f1').plot(marker='o', figsize=(8,4), title='H0 vs OVR EB LR')
plt.ylabel('Macro F1'); plt.tight_layout(); plt.show()
classes.sort_values('f1_delta').set_index('class').f1_delta.plot.barh(figsize=(8,7), title='OVR class F1 delta vs H0')
plt.tight_layout(); plt.show()
print('Recovered:', int(errors.recovered_by_ovr.sum()), 'Broken:', int(errors.broken_by_ovr.sum()))